<a href="https://colab.research.google.com/github/G0rav/machine_learning_explained_visually_free/blob/main/04-gradient-descent-linear-regression/notebook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Gradient Descent for Linear Regression

**[Watch the video](https://youtu.be/eycqI9ifm54)** · video 4 of *Machine Learning Explained
Visually*.

Video 3 built gradient descent on functions somebody chose. This one points it
at a model, on a real dataset, and then deals with the bill: the gradient of a
loss is a sum with one term per training row, so **one honest update costs a
full pass over all your data**.

Everything from the video, runnable. Three layers:

1. **Follow along** — every run in the video, in code, in the same order.
2. **Experiment** — break it on purpose and find out exactly where it breaks.
3. **Challenge** — write the pieces yourself before the next video.

Nothing here imports from anywhere. `numpy` is required. `scikit-learn` and
`matplotlib` are optional, and every cell that uses either says so and keeps
working without it. Run it top to bottom.

Every section asserts the number the video put on screen. If a check ever
fails, the video and this notebook have drifted apart and one of them is
wrong.

> **New here?** Video 3's notebook builds the update rule itself — the sign, the
> step that shrinks, the learning rate and the two ways of choosing it badly.
> This notebook assumes that rule and spends its time on what a *dataset* does
> to it.

---

# Layer 1 — Follow along

In [ ]:
import numpy as np

PASSED = 0
def check(label, got, want, tol=1e-9):
    """Assert a value the video states, and say so."""
    global PASSED
    got_f, want_f = float(got), float(want)
    assert abs(got_f - want_f) <= tol, f"{label}: {got_f!r} != {want_f!r}"
    PASSED += 1
    print(f"ok   {label:52s} {got_f:.9g}")

def check_all(label, got, want, tol=1e-9):
    """The same, for an array of them."""
    global PASSED
    got_a, want_a = np.asarray(got, float), np.asarray(want, float)
    worst = float(np.max(np.abs(got_a - want_a)))
    assert worst <= tol, f"{label}: worst difference {worst!r}\n{got_a}\n{want_a}"
    PASSED += 1
    print(f"ok   {label:52s} worst difference {worst:.3g}")

## Part 9 — Gradient descent for linear regression

The rule from video 3, unchanged:

$$x_{i+1} = x_i - r \cdot f'(x_i)$$

Point it at a model and two things get new names and nothing else changes. The
function being minimised is a **loss** $L$, and the variable being minimised
over is a **weight vector** $w$:

$$w_{j+1} = w_j - r \cdot \nabla L(w_j)$$

The loss for linear regression, with no regularisation and no intercept so the
algebra stays readable:

$$L(w) = \sum_{i=1}^{n} \left( y_i - w^\top x_i \right)^2$$

**What is the variable here?** This is the sentence the whole video turns on.
$x_i$ and $y_i$ are your training data. They are given. They never move. The
only thing you get to change is $w$ — so $L$ is a function of $w$, and you
differentiate with respect to $w$, not with respect to $x$.

Differentiating: the outside is *something squared*, which gives two times that
something; the chain rule then wants the derivative of the inside, and the
inside is $y_i - w^\top x_i$, whose derivative with respect to $w$ is $-x_i$
(video 2: $\frac{\partial}{\partial x} a^\top x = a$). So

$$\nabla L(w) = \sum_{i=1}^{n} -2 x_i \left( y_i - w^\top x_i \right)$$

Now the six cars from video 1 — age in years, price in thousands — and the
model $\text{price} = w \cdot \text{age} + b$. Carrying the intercept $b$ as a
separate number is easier than a column of ones on six rows, so the gradient
has two components, each the sum above written out for this model.

In [ ]:
AGE   = np.array([1.0, 2.0, 4.0, 6.0, 8.0, 9.0])
PRICE = np.array([23.2, 19.6, 15.2, 11.2, 7.6, 7.2])
N = len(AGE)

def residuals(w, b, idx=None):
    """price - (w*age + b), row by row: the bracket in the loss."""
    i = np.arange(N) if idx is None else np.asarray(idx)
    return PRICE[i] - (w * AGE[i] + b)

def loss(w, b):
    """L(w, b) = sum over the six cars of (price - (w*age + b))^2."""
    r = residuals(w, b)
    return float(r @ r)

def row_terms(w, b):
    """Each car's own contribution to the gradient: (-2*age*res, -2*res)."""
    r = residuals(w, b)
    return np.stack([-2.0 * AGE * r, -2.0 * r], axis=1)

def grad(w, b):
    """(dL/dw, dL/db) -- which is row_terms summed, because the gradient IS a sum."""
    return row_terms(w, b).sum(axis=0)

Start somewhere deliberately bad: `w = -0.5`, `b = 18`. That line is far too
flat for this data.

In [ ]:
W0, B0 = -0.5, 18.0

check("part 9  L at the bad start", loss(W0, B0), 134.98, 1e-9)

gw, gb = grad(W0, B0)
check("part 9  dL/dw there", gw, 246.0, 1e-9)
check("part 9  dL/db there", gb,  18.0, 1e-9)

**Those two numbers should look familiar.** Video 1 got `246` and `18` by
*measuring*: change `w` by a thousandth, see what the total does, divide. Here
they came out of a formula, by differentiating the loss. That is what videos 2
and 3 were for — and the two routes agreeing is the check worth running.

In [ ]:
H = 0.001   # video 1's nudge
measured_w = (loss(W0 + H, B0) - loss(W0, B0)) / H
measured_b = (loss(W0, B0 + H) - loss(W0, B0)) / H

print(f"     measured by nudging   dL/dw = {measured_w:.4f}   dL/db = {measured_b:.4f}")
print(f"     from the formula      dL/dw = {gw:.4f}   dL/db = {gb:.4f}")
check("part 9  the nudge agrees with the derivative in w", measured_w, gw, 0.3)
check("part 9  the nudge agrees with the derivative in b", measured_b, gb, 0.02)
print("     (the nudge is a difference quotient, so it is close and not equal --")
print("      that gap is exactly what video 2 removed)")

Now run it. `r = 0.002`, two thousand updates, and nothing in the loop except
the rule.

In [ ]:
R, STEPS = 0.002, 2000

def descend_cars(w0=W0, b0=B0, r=R, steps=STEPS):
    """Full-batch gradient descent. Returns every w, every b, every loss."""
    ws, bs = [float(w0)], [float(b0)]
    for _ in range(steps):
        gw, gb = grad(ws[-1], bs[-1])
        ws.append(ws[-1] - r * gw)
        bs.append(bs[-1] - r * gb)
    ws, bs = np.array(ws), np.array(bs)
    return ws, bs, np.array([loss(w, b) for w, b in zip(ws, bs)])

WS, BS, LOSSES = descend_cars()

MARKS = (0, 1, 5, 25, 100, 500, 2000)
SPOKEN = (134.98, 63.27, 58.37, 46.77, 21.51, 4.61, 4.48)
print(f"{'updates':>8}  {'L':>10}")
for k, want in zip(MARKS, SPOKEN):
    print(f"{k:8d}  {LOSSES[k]:10.2f}")
check_all("part 9  the loss at every mark the video reads out",
          [round(float(LOSSES[k]), 2) for k in MARKS], SPOKEN, 0.0)

In [ ]:
END_W, END_B = float(WS[-1]), float(BS[-1])
check("part 9  w after 2,000 updates", END_W, -1.9999950578319678, 1e-12)
check("part 9  b after 2,000 updates", END_B, 23.999966971651858, 1e-12)
check("part 9  and the loss there",    LOSSES[-1], 4.480000001685184, 1e-9)

print(f"\n     price = {END_B:.6f} - {abs(END_W):.6f} * age")
print(f"     which to the precision anyone cares about is  price = 24 - 2 * age")
print("     -- the same line video 1 found, with nothing measured and nothing solved")

## Part 10 — What one update costs

Look at the gradient again, because there is a problem hiding in it:

$$\nabla L(w) = \sum_{i=1}^{n} -2 x_i \left( y_i - w^\top x_i \right)$$

The summation runs over **every row**. And you need the gradient at *every*
update, at a different `w` each time, so nothing can be reused. One update is
one full pass over the data.

On six cars that is free. The cost is not about six cars.

In [ ]:
N_SMALL, N_LARGE = N, 1_000_000
UPDATES = STEPS                       # the 2,000 we just ran

visits_small = N_SMALL * UPDATES
visits_large = N_LARGE * UPDATES

check("part 10  row visits, six cars, 2,000 updates", visits_small, 12_000, 0)
check("part 10  row visits, a million rows",          visits_large, 2_000_000_000, 0)

print(f"\n     {N_SMALL:>9,} rows x {UPDATES:,} updates = {visits_small:>15,} row visits")
print(f"     {N_LARGE:>9,} rows x {UPDATES:,} updates = {visits_large:>15,} row visits")
print("\n     Same algorithm, same number of updates. The second one is the")
print("     reason this part exists -- and it is not specific to linear")
print("     regression: differentiate any loss that is a sum over your data")
print("     and you get a gradient that is a sum over your data.")

## Part 11 — Stochastic gradient descent

The fix is one character. Don't sum over $n$; sum over $k$:

$$w_{j+1} = w_j - r \sum_{i \in S_j} -2 x_i \left( y_i - w_j^\top x_i \right),
\qquad |S_j| = k \le n$$

with one condition that matters: **you draw a fresh $S_j$ every iteration.**
Pick a subset once and keep reusing it and you are not doing this any more, you
are training on a smaller dataset.

Why would that work? Because the full gradient *is* the sum of the per-row
pieces, so a single row is one of the things being added up. Here are all six.

In [ ]:
ROWS0 = row_terms(W0, B0)
print(f"{'age':>5}  {'dL/dw piece':>14}  {'dL/db piece':>14}")
for age, (rw, rb) in zip(AGE, ROWS0):
    print(f"{age:5.0f}  {rw:14.1f}  {rb:14.1f}")
print(f"{'sum':>5}  {ROWS0[:, 0].sum():14.1f}  {ROWS0[:, 1].sum():14.1f}")

check_all("part 11  the six pieces, exactly as spoken",
          ROWS0, [[-11.4, -11.4], [-10.4, -5.2], [6.4, 1.6],
                  [45.6, 7.6], [102.4, 12.8], [113.4, 12.6]], 1e-9)
check_all("part 11  and they add up to the full gradient",
          ROWS0.sum(axis=0), grad(W0, B0), 1e-12)

**Now the part that is usually skipped.** Those six pieces are not all pointing
the same way. Measure the angle between each one and the total.

In [ ]:
FULL = grad(W0, B0)

def angle_between(u, v):
    """Degrees between two 2-vectors."""
    u, v = np.asarray(u, float), np.asarray(v, float)
    cos = float(u @ v) / (float(np.linalg.norm(u)) * float(np.linalg.norm(v)))
    return float(np.degrees(np.arccos(np.clip(cos, -1.0, 1.0))))

ANGLES = np.array([angle_between(row, FULL) for row in ROWS0])
BACKWARD = np.where(ANGLES > 90.0)[0]

for age, a in zip(AGE, ANGLES):
    flag = "  <-- backwards" if a > 90 else ""
    print(f"     the {age:.0f} year old car:  {a:6.1f} degrees from the full gradient{flag}")

check_all("part 11  the two obtuse angles, as spoken",
          [int(round(ANGLES[i])) for i in BACKWARD], [139, 158], 0)
check("part 11  how many of the six point backwards", len(BACKWARD), 2, 0)

Two of the six point *backwards*. So when the algorithm draws one of those, that
update moves `w` the wrong way — and "wrong way" is not a figure of speech
here: take the step and the loss goes **up**.

In [ ]:
R_SGD = 0.0005
step_losses = np.array([loss(W0 - R_SGD * gw_i, B0 - R_SGD * gb_i)
                        for gw_i, gb_i in ROWS0])
rose = np.where(step_losses > loss(W0, B0))[0]

print(f"     L before any step: {loss(W0, B0):.4f}\n")
for age, L in zip(AGE, step_losses):
    arrow = "UP  " if L > loss(W0, B0) else "down"
    print(f"     one step on the {age:.0f} year old car -> L = {L:9.4f}   ({arrow})")

check("part 11  how many single-row steps raise the loss", len(rose), 2, 0)
check_all("part 11  and they are the same two rows", rose, BACKWARD, 0)
print("\n     The two that raise the loss are exactly the two that point backwards.")
print("     Nothing here is read off a picture.")

It still arrives, because you are not taking one step, you are taking thousands
on freshly drawn rows, and on average the direction you move is the direction
the full gradient points.

Here it is run for real: `k = 1`, `r = 0.0005`, twenty thousand draws.

In [ ]:
K, DRAWS, SEED = 1, 20_000, 7

def sgd(w0=W0, b0=B0, r=R_SGD, draws=DRAWS, k=K, seed=SEED):
    """The k-row update, redrawn every iteration. The redraw is INSIDE the loop."""
    rng = np.random.default_rng(seed)
    w, b = float(w0), float(b0)
    ws, bs = [w], [b]
    for _ in range(draws):
        idx = rng.integers(0, N, size=k)          # fresh rows, every time
        res = residuals(w, b, idx)
        w -= r * float(-2.0 * (AGE[idx] * res).sum())
        b -= r * float(-2.0 * res.sum())
        ws.append(w)
        bs.append(b)
    return np.array(ws), np.array(bs)

SGD_WS, SGD_BS = sgd()
SGD_W, SGD_B = float(SGD_WS[-1]), float(SGD_BS[-1])
SGD_LOSS = loss(SGD_W, SGD_B)

check("part 11  w after 20,000 single-row draws", SGD_W, -1.9650244205414313, 1e-12)
check("part 11  b after 20,000 single-row draws", SGD_B, 23.985448980965927, 1e-12)
check("part 11  the loss it reached",             SGD_LOSS, 4.697839387593276, 1e-12)

**And now be honest about it.** On six rows, this is a worse deal, and the video
says so out loud.

In [ ]:
print(f"     full batch :  {STEPS:>6,} updates   L = {LOSSES[-1]:.4f}")
print(f"     k = 1      :  {DRAWS:>6,} updates   L = {SGD_LOSS:.4f}")
check("part 11  iterations, stochastic over batch", DRAWS // STEPS, 10, 0)
assert SGD_LOSS > LOSSES[-1]
PASSED += 1
print(f"ok   {'part 11  ten times the iterations, worse answer':52s} on six rows")

print("\n     Of course it is. Reading all six rows was free, so there was")
print("     nothing to save. The argument is about the million.")

In [ ]:
BIG_N, BIG_K = 1_000_000, 100
cost_ratio = BIG_N // BIG_K
check("part 11  rows per iteration, batch over k=100", cost_ratio, 10_000, 0)

print(f"     n = {BIG_N:,}, k = {BIG_K}")
print(f"     each iteration reads {cost_ratio:,}x fewer rows")
print(f"     at 10x the iterations that is still {cost_ratio // 10:,}x less work")
print("\n     Vocabulary: k is the BATCH SIZE. k = 1 is usually called")
print(f"     stochastic gradient descent; 1 < k << n is mini-batch, and {BIG_K}")
print("     or 256 are common. People use the terms loosely -- read the k.")

## Part 12 — Writing it

Three programs. The first is video 3's whole algorithm, on
$f(x) = x^2 - 3x + 2$, in six lines.

In [ ]:
def f(x):   return x ** 2 - 3 * x + 2
def df(x):  return 2 * x - 3

x, r, tol = 0.0, 0.3, 1e-6
updates = 0
while True:
    step = r * df(x)
    x -= step
    updates += 1
    if abs(step) < tol:
        break

check("part 12  updates to stop at tol 1e-6", updates, 16, 0)
check("part 12  and where it stopped",        x, 1.4999993557549056, 1e-15)
print("     Six lines, and there is nothing else in it.")

The second is the same loop on the six cars. **The only thing that changes is
what the gradient is** — four lines of arithmetic instead of one.

In [ ]:
w, b, r = -0.5, 18.0, 0.002
for _ in range(2000):
    res = PRICE - (w * AGE + b)
    gw = -2 * (AGE * res).sum()
    gb = -2 * res.sum()
    w, b = w - r * gw, b - r * gb

check("part 12  the car loop's w", w, -1.9999950578319678, 1e-12)
check("part 12  the car loop's b", b, 23.999966971651858, 1e-12)
print(f"     w = {round(w, 6)}, b = {round(b, 6)}  -- part 9's run, in four lines")

The third is the library. `SGDRegressor` is the **stochastic** one — the name
says so — and every option is set explicitly rather than defaulted, because the
defaults quietly change the problem: the default penalty adds a regularisation
term our loss does not have, and the default learning rate schedule changes `r`
as it goes.

This cell runs the real thing if `scikit-learn` is installed and otherwise uses
the fitted results, which are the same numbers either way.

In [ ]:
SK_ETA0, SK_TOL, SK_MAX_ITER, SK_SEED = 0.002, 1e-9, 50_000, 1
SK_BIG_ETA0 = 0.02

# Fitted with scikit-learn and frozen, so this cell states a number even where
# scikit-learn is not installed. If it IS installed, they are refitted and the
# checks below compare the two.
FROZEN = {
    0.002: (-1.995452, 23.957804, 2282),
    0.02:  (-1.766528, 24.007617, None),
}
FROZEN_LSTSQ = (-1.9999999999999993, 23.999999999999996)

try:
    from sklearn.linear_model import LinearRegression, SGDRegressor
    HAVE_SK = True
except ImportError:
    HAVE_SK = False
    print("scikit-learn not installed -- using the fitted results below.")
    print("`pip install scikit-learn` to refit them yourself; nothing else needs it.\n")

def fit_sgd(eta0):
    """(w, b, epochs) from SGDRegressor, or the frozen fit of the same call."""
    if not HAVE_SK:
        return FROZEN[eta0]
    m = SGDRegressor(loss="squared_error", penalty=None,
                     learning_rate="constant", eta0=eta0,
                     max_iter=SK_MAX_ITER, tol=SK_TOL, random_state=SK_SEED)
    m.fit(AGE.reshape(-1, 1), PRICE)
    return round(float(m.coef_[0]), 6), round(float(m.intercept_[0]), 6), int(m.n_iter_)

SK_W, SK_B, SK_EPOCHS = fit_sgd(SK_ETA0)
check("part 12  SGDRegressor's w",      SK_W, -1.995452, 1e-9)
check("part 12  SGDRegressor's b",      SK_B, 23.957804, 1e-9)
check("part 12  SGDRegressor's epochs", SK_EPOCHS, 2282, 0)
check("part 12  the loss it reached",   round(loss(SK_W, SK_B), 6), 4.483347, 1e-9)

Compare that with our own loop. The library's answer is off **in the third
decimal place** — and the place is computed from the two numbers, not counted
by eye.

In [ ]:
GAP = abs(SK_W - w)
PLACE = int(np.ceil(-np.log10(GAP)))

print(f"     our loop      w = {w:.6f}")
print(f"     SGDRegressor  w = {SK_W:.6f}")
print(f"     gap           {GAP:.3e}")
check("part 12  the decimal place they first differ in", PLACE, 3, 0)

**That is not a bug in either of them.** It is the single-row update showing up
on its own from somebody else's code: near the minimum the run does not settle
onto a point, it keeps moving around near one, and where it stops is wherever it
happened to be when the tolerance fired.

You can make it worse on purpose. Leave everything alone and push `eta0` up ten
times.

In [ ]:
BIG_W, BIG_B, _ = fit_sgd(SK_BIG_ETA0)
check("part 12  SGDRegressor at eta0 = 0.02, w", BIG_W, -1.766528, 1e-9)
check("part 12  SGDRegressor at eta0 = 0.02, b", BIG_B, 24.007617, 1e-9)

assert abs(BIG_W - (-2.0)) < 0.5, "it settles loosely; it does not diverge"
PASSED += 1
print(f"ok   {'part 12  still the right line, just further from it':52s} w = {BIG_W}")

And here is the claim behind that, measured rather than asserted. `SGDRegressor`
minimises $\tfrac12 (y - \hat y)^2$ per sample, so its update is
`w += eta0 * residual * x` with no factor of two. Run *that* rule, at both
rates, long enough to reach steady state, and look at the spread of `w` over the
tail — not at where it happened to stop.

In [ ]:
def sk_style_sgd(eta0, draws=40_000, seed=11, w0=W0, b0=B0):
    """scikit-learn's own per-sample update: w += eta0 * res * x, no 2."""
    rng = np.random.default_rng(seed)
    w_, b_ = float(w0), float(b0)
    ws, bs = [w_], [b_]
    for _ in range(draws):
        i = int(rng.integers(0, N))
        res = float(PRICE[i] - (w_ * AGE[i] + b_))
        w_ += eta0 * res * float(AGE[i])
        b_ += eta0 * res
        ws.append(w_)
        bs.append(b_)
    return np.array(ws), np.array(bs)

TAIL = 4_000
clouds = {eta: sk_style_sgd(eta) for eta in (SK_ETA0, SK_BIG_ETA0)}
spread = {eta: float(np.std(ws[-TAIL:])) for eta, (ws, _) in clouds.items()}
centre = {eta: float(np.mean(ws[-TAIL:])) for eta, (ws, _) in clouds.items()}

for eta in (SK_ETA0, SK_BIG_ETA0):
    print(f"     eta0 = {eta:<6}  centre {centre[eta]:+.4f}   spread {spread[eta]:.4f}")

check("part 12  spread at eta0 = 0.002", spread[SK_ETA0],     0.029692678760330684, 1e-15)
check("part 12  spread at eta0 = 0.02",  spread[SK_BIG_ETA0], 0.144914547454169, 1e-15)
check("part 12  how much wider the big rate orbits",
      spread[SK_BIG_ETA0] / spread[SK_ETA0], 4.880480761734921, 1e-12)

# -1.7665 is an ordinary member of the wide cloud and an impossible member of
# the tight one. That is the whole argument, in standard deviations.
wide_sd  = abs(BIG_W - centre[SK_BIG_ETA0]) / spread[SK_BIG_ETA0]
tight_sd = abs(BIG_W - centre[SK_ETA0])     / spread[SK_ETA0]
print(f"\n     -1.7665 is {wide_sd:.1f} sd from the centre of the WIDE cloud")
print(f"     -1.7665 is {tight_sd:.1f} sd from the centre of the TIGHT cloud")
assert wide_sd < 2.0 < 5.0 < tight_sd
PASSED += 1
print(f"ok   {'part 12  an ordinary draw at 0.02, an impossible one at 0.002':52s}")
print("\n     Both rates circle the same centre, near the exact answer.")
print("     The rate does not decide where you go. It decides how close you stay.")

One last check, and it is a satisfying one. `LinearRegression` on the same six
cars returns `-2` and `24` to fourteen decimal places — because it is not
iterating at all. It solves for the answer in one calculation. Part 13 comes
back to that.

In [ ]:
if HAVE_SK:
    fit = LinearRegression().fit(AGE.reshape(-1, 1), PRICE)
    LSTSQ_W, LSTSQ_B = float(fit.coef_[0]), float(fit.intercept_)
else:
    LSTSQ_W, LSTSQ_B = FROZEN_LSTSQ

PLACES = 14
check("part 12  LinearRegression's w", LSTSQ_W, -1.9999999999999993, 0.0)
check("part 12  LinearRegression's b", LSTSQ_B, 23.999999999999996, 0.0)
check("part 12  w, to the place the video claims", round(LSTSQ_W, PLACES), -2.0, 0.0)
check("part 12  b, to the place the video claims", round(LSTSQ_B, PLACES), 24.0, 0.0)
assert (LSTSQ_W, LSTSQ_B) != (-2.0, 24.0), "exact to 14 places is not exact"
PASSED += 1
print(f"ok   {'part 12  ...and not to the fifteenth, which is the honest bit':52s}")

## Part 13 — What you can now explain, and the one limit

Three sentences worth keeping:

1. **The loss is a function of the weights, not of the data.** The `x`s and `y`s
   are given and never move. That is what turns training a model into a
   minimisation problem you already knew how to solve.
2. **The gradient of that loss is a sum with one term per row**, so one honest
   update costs a full pass over your data. That is the number that decides what
   training costs.
3. **The rule is video 3's, unchanged**, with $\nabla L$ where $f'$ was.

And then the limit, stated plainly rather than left to be found out. Gradient
descent only ever reads the derivative **at the point it is standing on**. It
has no information about the function anywhere else. So on a function with more
than one minimum, where you start decides which one you reach.

Here is a curve with two. It is built from its *derivative* —
$d(x) = \tfrac12 (x+2)(x-0.4)(x-2)$, whose three roots are the two minima and
the maximum between them — so the stationary points are exact by construction.

In [ ]:
def two_minima(x):
    z = np.asarray(x, float)
    return 0.5 * (z ** 4 / 4.0 - 0.4 * z ** 3 / 3.0 - 2.0 * z ** 2 + 1.6 * z)

def two_minima_d(x):
    z = np.asarray(x, float)
    return 0.5 * (z ** 3 - 0.4 * z ** 2 - 4.0 * z + 1.6)

ROOTS = np.array([-2.0, 0.4, 2.0])
check_all("part 13  the derivative really is zero at all three",
          two_minima_d(ROOTS), [0.0, 0.0, 0.0], 1e-12)

DEPTHS = two_minima(ROOTS)
for x0, d in zip(ROOTS, DEPTHS):
    kind = "maximum" if x0 == 0.4 else "minimum"
    print(f"     x = {x0:+.1f}   f = {d:+.4f}   ({kind})")
check("part 13  the global minimum is the left one", ROOTS[int(np.argmin(DEPTHS))], -2.0)
check("part 13  how much worse the other minimum is",
      abs(DEPTHS[0] - DEPTHS[2]), 2.1333333333333337, 1e-12)

Two runs. Same function, same rule, same `r`. **The only difference is `x0`**,
and they start four tenths apart.

In [ ]:
R13, STEPS13 = 0.15, 24

def descend(d, x0, r=R13, steps=STEPS13):
    xs = [float(x0)]
    for _ in range(steps):
        xs.append(xs[-1] - r * float(d(xs[-1])))
    return np.array(xs)

runs = [descend(two_minima_d, x0) for x0 in (0.2, 0.6)]
ends = [float(run[-1]) for run in runs]

print(f"     started at 0.2  ->  {ends[0]:+.6f}")
print(f"     started at 0.6  ->  {ends[1]:+.6f}")
check("part 13  0.4 apart at the start", abs(0.6 - 0.2), 0.4, 1e-12)
check("part 13  4.0 apart at the end",   abs(ends[1] - ends[0]), 4.0, 1e-4)
print("\n     Four tenths apart at the start, four whole units apart at the end,")
print("     and one of them found the worse minimum. The algorithm cannot tell.")

**Our loss is not like that, and that is a property of this loss rather than a
promise about the next one.** Take the real squared-error loss of parts 9 to 12,
hold `b` at its answer, and look along `w`: one minimum, at `-2`, worth `4.48`.
That is why every run in this video landed in the same place.

In [ ]:
def loss_along_w(w_):
    """The real L(w, b=24), not a stand-in parabola."""
    return np.array([loss(float(v), 24.0) for v in np.atleast_1d(np.asarray(w_, float))])

grid = np.linspace(-3.2, -0.8, 2401)
curve = loss_along_w(grid)
turning = grid[np.where(np.diff(np.sign(np.diff(curve))) != 0)[0] + 1]

check("part 13  the car loss along w has exactly one turning point", len(turning), 1, 0)
check("part 13  and it is at w = -2",   turning[0], -2.0, 1e-3)
check("part 13  worth",                 loss(-2.0, 24.0), 4.48, 1e-9)

And the thing that turned up at the end of part 12. We ran two thousand updates
to get to `-2` and `24`. `LinearRegression` got the same two numbers without
iterating once — because for linear regression you can set the gradient to zero
and *solve*. The loss is quadratic in `w`, so that equation is linear, so it has
one answer and you can write it down.

That derivation is a good one to do yourself and the rules from videos 2 and 3
are all it needs. Here it is, so you can check yours.

In [ ]:
# Set both components of the gradient to zero and solve the 2x2 system.
# dL/dw = 0 and dL/db = 0  =>  the normal equations.
X = np.stack([AGE, np.ones(N)], axis=1)       # the column of ones IS the intercept
direct_w, direct_b = np.linalg.solve(X.T @ X, X.T @ PRICE)

print(f"     2,000 updates    w = {END_W:.12f}   b = {END_B:.12f}")
print(f"     one calculation  w = {direct_w:.12f}   b = {direct_b:.12f}")
check("part 13  solving directly gives w", round(float(direct_w), 12), -2.0, 0.0)
check("part 13  solving directly gives b", round(float(direct_b), 12), 24.0, 0.0)
print("\n     So why iterate at all? Because 'set the gradient to zero and solve'")
print("     only works when you CAN solve it -- video 3's part 1 and part 8 are")
print("     about the case where you cannot, which is almost everything else.")

In [ ]:
print(f"Layer 1 complete — {PASSED} checks passed, all of them numbers the video states.")

---

# Layer 2 — Experiment

The video shows `k = 1` against full batch on six rows, and two learning rates
in the library. Those are points. Here are the lines they sit on.

## The batch size, all the way from 1 to n

`k` is the only new knob this video adds. Sweep it, at a fixed number of
*iterations*, and two things show up: bigger `k` gets closer, and bigger `k`
costs proportionally more rows to get there. The last column is the one that
matters — it is the bill.

In [ ]:
ITERS = 6_000
print(f"{'k':>3}  {'final w':>10}  {'final b':>10}  {'loss':>9}  {'rows read':>11}")
for k in (1, 2, 3, 4, 6):
    ws_k, bs_k = sgd(r=R_SGD, draws=ITERS, k=k, seed=7)
    wk, bk = float(ws_k[-1]), float(bs_k[-1])
    print(f"{k:3d}  {wk:10.5f}  {bk:10.5f}  {loss(wk, bk):9.4f}  {k * ITERS:11,}")

print(f"\n{'batch':>3}  {'':10}  {'':10}  {LOSSES[-1]:9.4f}  {N * STEPS:11,}")
print("\nk = 6 IS full batch on six rows, at six times the row cost of k = 1.")
print("On six rows that is the whole story and it is the wrong story --")
print("see the next cell for the one that is not.")

## The bill, at a size where it matters

The comparison the video actually makes is between *algorithms at a fixed
budget of rows read*, not at a fixed number of iterations. Assume the pessimistic
version of the stochastic case — it needs ten times the iterations — and price
both.

In [ ]:
print(f"{'n':>12}  {'batch rows':>16}  {'k=100 rows':>16}  {'saving':>10}")
for n in (6, 1_000, 100_000, 1_000_000, 100_000_000):
    k = min(100, n)
    batch_rows = n * UPDATES
    sgd_rows = k * UPDATES * 10          # ten times the iterations
    print(f"{n:12,}  {batch_rows:16,}  {sgd_rows:16,}  {batch_rows / sgd_rows:9.1f}x")

print("\nThe crossover is around n = 1,000. Below it, full batch wins and the")
print("six cars are not a counter-example to anything. Above it, the gap is")
print("the reason every large model is trained this way.")

## The learning rate decides how close you stay

This is the claim part 12 ends on, and it is the one thing that is genuinely
different about the stochastic version. With full batch, a smaller `r` just
means slower. With `k = 1`, `r` also sets the **radius of the orbit** the run
never leaves.

In [ ]:
print(f"{'eta0':>8}  {'centre of w':>13}  {'spread of w':>13}  {'x wider':>9}")
base = None
for eta in (0.0005, 0.001, 0.002, 0.005, 0.01, 0.02, 0.05):
    ws_e, _ = sk_style_sgd(eta, draws=40_000)
    sd = float(np.std(ws_e[-TAIL:]))
    mid = float(np.mean(ws_e[-TAIL:]))
    base = sd if base is None else base
    print(f"{eta:8.4f}  {mid:+13.5f}  {sd:13.5f}  {sd / base:8.1f}x")

print("\nThe centre barely moves. The spread grows with eta0, roughly linearly.")
print("That is why the library's -1.7665 is not a worse answer -- it is the")
print("same answer, sampled from a wider orbit.")

## Where you start, on a loss that has two minima

Part 13 runs two starting points. Here is the whole basin map: for every `x0`
on a grid, which minimum does it end in? The boundary is the maximum at `0.4`,
exactly — and *nothing in the algorithm knows that*.

In [ ]:
starts = np.linspace(-1.0, 1.6, 27)
landed = np.array([descend(two_minima_d, x0, steps=200)[-1] for x0 in starts])
left = landed < 0

boundary_lo = starts[left][-1]
boundary_hi = starts[~left][0]
print(f"     every start below {boundary_lo:+.3f} ends at -2 (the global minimum)")
print(f"     every start above {boundary_hi:+.3f} ends at +2 (the local one)")
print(f"     the maximum sits at +0.400, between them")
assert boundary_lo < 0.4 < boundary_hi
print("\n     Move x0 across a point the algorithm has never evaluated and the")
print("     answer changes by four units. That is the limit, drawn.")

## Optional: see it (needs `matplotlib`)

In [ ]:
try:
    import matplotlib.pyplot as plt

    fig, axes = plt.subplots(1, 3, figsize=(14, 3.8))

    # 1. the fit, before and after
    ax = axes[0]
    ax.scatter(AGE, PRICE, zorder=3)
    span = np.linspace(0, 10, 2)
    ax.plot(span, W0 * span + B0, "--", lw=1, label=f"start  L = {loss(W0, B0):.0f}")
    ax.plot(span, END_W * span + END_B, lw=2, label=f"fitted L = {LOSSES[-1]:.2f}")
    ax.set_xlabel("age (years)"); ax.set_ylabel("price (thousands)")
    ax.set_title("2,000 updates"); ax.legend(fontsize=8)

    # 2. batch against stochastic, in parameter space
    ax = axes[1]
    ax.plot(WS, BS, lw=2, label="full batch")
    ax.plot(SGD_WS[::20], SGD_BS[::20], lw=0.6, alpha=0.7, label="k = 1")
    ax.plot([-2], [24], "*", ms=14, label="the exact answer")
    ax.set_xlabel("w"); ax.set_ylabel("b"); ax.set_title("the path each one takes")
    ax.legend(fontsize=8)

    # 3. the two orbits, zoomed in on the minimum
    ax = axes[2]
    for eta in (SK_ETA0, SK_BIG_ETA0):
        ws_e, _ = clouds[eta]
        ax.plot(ws_e[-TAIL:], lw=0.5, label=f"eta0 = {eta}")
    ax.axhline(-2.0, ls="--", lw=1, color="k")
    ax.set_xlabel("draw"); ax.set_ylabel("w")
    ax.set_title("it never settles, it orbits"); ax.legend(fontsize=8)

    fig.tight_layout()
    plt.show()
except ImportError:
    print("matplotlib not installed — skipping the pictures, nothing else needs them")

---

# Layer 3 — Challenge

Four exercises, in the order the rest of the course needs them. Each has a check
you can run against your own answer.

**1. The gradient, from the formula.** Write the two components for
$L(w,b) = \sum (\text{price}_i - (w \cdot \text{age}_i + b))^2$ yourself. This is
the one derivation the whole video rests on, and it is two lines.

In [ ]:
def my_grad(w, b):
    # replace with your own answer -- return (dL/dw, dL/db)
    return None

if my_grad(-0.5, 18.0) is None:
    print("not attempted yet — edit my_grad above")
    print("hint: differentiate (price - (w*age + b))^2 with respect to w, keep")
    print("      the sum, and remember the chain rule brings down a -age.")
else:
    ok = True
    for probe in ((-0.5, 18.0), (0.0, 0.0), (-2.0, 24.0), (1.5, -3.0)):
        got, want = np.asarray(my_grad(*probe), float), grad(*probe)
        good = np.allclose(got, want, atol=1e-9)
        ok &= good
        print(f"{'ok  ' if good else 'FAIL'} at w={probe[0]:>5}, b={probe[1]:>5}   "
              f"yours {np.round(got, 4)}   reference {np.round(want, 4)}")
    print("\nall four agree" if ok else "\nnot yet — check the sign and the factor of 2")

**2. Write the stochastic loop.** `k` rows, drawn fresh every iteration. The
redraw has to be *inside* the loop — that is the whole condition — and if you
use `np.random.default_rng(7)` and `rng.integers(0, 6, size=k)` in that order
you will reproduce the video's run exactly.

In [ ]:
def my_sgd(w0, b0, r, draws, k, seed):
    # replace with your own answer -- return (w, b) after `draws` iterations
    return None

if my_sgd(-0.5, 18.0, 0.0005, 20_000, 1, 7) is None:
    print("not attempted yet — edit my_sgd above")
else:
    got = np.asarray(my_sgd(-0.5, 18.0, 0.0005, 20_000, 1, 7), float)
    want = np.array([SGD_W, SGD_B])
    print(f"yours     w = {got[0]:.9f}, b = {got[1]:.9f}")
    print(f"the video w = {want[0]:.9f}, b = {want[1]:.9f}")
    if np.allclose(got, want, atol=1e-9):
        print("ok — identical, which means you drew the same rows in the same order")
    elif abs(loss(*got) - loss(*want)) < 0.5:
        print("close — right algorithm, different draws. Check the rng and the redraw.")
    else:
        print("not yet — is the draw inside the loop?")

**3. Solve it instead.** Set both components of the gradient to zero and solve
for `w` and `b` by hand — two equations, two unknowns, no loop and no
`np.linalg.solve`. You will need $\sum \text{age}$, $\sum \text{age}^2$,
$\sum \text{price}$ and $\sum \text{age}\cdot\text{price}$, and nothing else.

In [ ]:
def my_exact_fit():
    # replace with your own answer -- return (w, b), computed from the four sums
    return None

if my_exact_fit() is None:
    print("not attempted yet — edit my_exact_fit above")
    print("hint: dL/db = 0 gives sum(price) = w*sum(age) + n*b, which is one of")
    print("      the two equations. dL/dw = 0 gives the other. Then eliminate b.")
else:
    got = np.asarray(my_exact_fit(), float)
    print(f"yours   w = {got[0]:.12f}, b = {got[1]:.12f}")
    print(f"target  w = -2, b = 24, to about fourteen places")
    print("ok" if np.allclose(got, [-2.0, 24.0], atol=1e-10)
          else "not yet — check which sum multiplies which")

**4. Before the next video: `w` as a vector.** Everything above carried `w` and
`b` as two separate numbers because six cars did not need anything more. The
general form does not have a `b` at all — the whole prediction is
$w^\top x$ — and you get the intercept back by adding a **column of ones**,
because then one entry of `w` is multiplied by `1` in every row.

Do that. One `w`, no `b`, and the same update rule with the gradient written as
a single matrix expression:

$$\nabla L(w) = -2 X^\top (y - Xw)$$

In [ ]:
X_DESIGN = np.stack([AGE, np.ones(N)], axis=1)    # n x 2: [age, 1]

def my_vector_fit(w0=np.array([-0.5, 18.0]), r=0.002, steps=2000):
    # replace with your own answer -- return w, a length-2 vector, where w[1]
    # is the intercept because that is the column of ones
    return None

if my_vector_fit() is None:
    print("not attempted yet — edit my_vector_fit above")
    print("hint: residual = y - X @ w, and grad = -2 * X.T @ residual.")
    print("      One line each. No loop over rows anywhere.")
else:
    got = np.asarray(my_vector_fit(), float)
    print(f"yours        w = {got[0]:.9f}, intercept = {got[1]:.9f}")
    print(f"part 9's run w = {END_W:.9f}, intercept = {END_B:.9f}")
    print("ok — same answer, and the loop no longer knows how many features there are"
          if np.allclose(got, [END_W, END_B], atol=1e-9)
          else "not yet — check the transpose and the factor of 2")

---

That is video 4: the same rule as last time, pointed at a loss instead of a
function somebody chose, and the one fact that changes everything about the
cost — **the gradient is a sum over your data, so an honest update reads all of
it.** When that is too expensive, you sum over a random handful and redraw it
every iteration, and that is stochastic gradient descent, which is what actually
trains everything.

Two things it cannot do, which you should now be able to state without being
told: it cannot tell a local minimum from the global one, because it only ever
reads the derivative where it stands; and it will happily spend two thousand
updates on a problem you could have solved in one calculation, if you did not
notice that you could.

**[Watch the video](https://www.youtube.com/@school_whool)** · and if anything
here did not land, put it in the comments.